# End-to-End Reproduction Notebook

This notebook allows you to reproduce the project's results using cached data, running the full pipeline from video embedding to analysis without external API costs.

## Cross-Platform Compatibility
This notebook is designed to run on **Windows**, **Linux**, and **macOS**.

## Steps:
1.  **Setup**: Install dependencies & download necessary data.
2.  **Asset Generation**: Generate (or verify) video embeddings.
3.  **Experiment Execution**: Run experiments using the cached LLM responses.
4.  **Analysis**: Visualize and compare results.

In [ ]:
import os
import sys
from pathlib import Path

# IMPORTANT: Ensure we are running from the project root
# If running from 'notebooks/', move up one level.
if Path.cwd().name == 'notebooks':
    os.chdir('..')

print(f"Working Directory: {Path.cwd()}")

# Add project root to sys.path so we can import 'src' modules directly if needed
if str(Path.cwd()) not in sys.path:
    sys.path.append(str(Path.cwd()))

# Also add 'src' to sys.path to support imports like 'from data...'
if str(Path.cwd() / "src") not in sys.path:
    sys.path.append(str(Path.cwd() / "src"))

## 1. Setup Environment & Data
First, we ensure all Python libraries are installed and the data is downloaded.

In [ ]:
import subprocess

# 1.1 Install system dependencies
print("Installing dependencies from requirements.txt...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])

print("Installing project in editable mode...")
# Installs the project root (current dir) in editable mode
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", "."])

print("Dependencies installed.")

In [ ]:
# 1.2 Download Data
# We run the script using the same python executable
# We run relative to project root
print("Running download_data.py...")
subprocess.check_call([sys.executable, "scripts/download_data.py"])

## 2. Generate Video Embeddings
This step processes videos in `datasets/wild_videos` and saves embeddings to `local/wild_videos_embs`.
Since we downloaded the cache, most of these should already exist and be skipped.

**Note:** Video processing requires `opencv` and `timm`, which can conflict with `numpy` 2.x versions used in analysis. 
If embeddings are missing, this cell will automatically install the necessary incompatible versions, run the generation in a subprocess, and then revert the environment.

In [ ]:
# Define paths relative to valid project root
# Path construction is cross-platform
video_dir = Path("datasets") / "wild_videos"
output_dir = Path("local") / "wild_videos_embs"

if output_dir.exists() and any(output_dir.iterdir()):
    print(f"Embeddings found in {output_dir}. Skipping generation.")
else:
    print(f"Embeddings NOT found in {output_dir}.")
    print("Starting transient environment setup for video processing...")
    
    try:
        # 1. Install temporary conflicting dependencies
        print("Step 1/3: Installing video dependencies (opencv-python-headless, timm, numpy<2)...")
        # We force upgrade to resolve conflicts between existing torch and requested timm/torchvision
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "opencv-python-headless", "timm", "numpy<2", "torch", "torchvision"])
        
        # 2. Run generation in a subprocess
        # We use a subprocess to rely on the newly installed package versions on disk,
        # without confusing the currently running notebook kernel (which has numpy 2.x loaded).
        print("Step 2/3: Generating embeddings (this may take a while)...")
        
        # We pass absolute paths to avoid CWD ambiguity in the subprocess code snippet
        project_root = Path.cwd()
        
        script_code = f"""
import sys
from pathlib import Path
# Ensure we can import src
sys.path.append(str(Path('{str(project_root).replace('\\', '/')}')))

from src.data.video_embeddings import VideoEmbedder

video_dir = Path('{str(project_root / video_dir).replace('\\', '/')}')
output_dir = Path('{str(project_root / output_dir).replace('\\', '/')}')

print(f'Importing VideoEmbedder...')
embedder = VideoEmbedder()
print(f'Processing...')
embedder.process_directory(
    video_dir=video_dir,
    output_dir=output_dir,
    fps=1,
    clip_size=1
)
"""
        subprocess.check_call([sys.executable, "-c", script_code])
        
        print("Generation complete.")
        
    except subprocess.CalledProcessError as e:
        print(f"ERROR during generation: {e}")
        raise e
        
    finally:
        # 3. Restore environment
        print("Step 3/3: Restoring original environment...")
        # Uninstall video libs to be clean
        subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "opencv-python-headless", "timm"])
        # Force reinstall of requirements to fix numpy and torch versions
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
        print("Environment restored.")

## 3. Run Experiments
We will run the specified configurations. The `--cached-execution-only` (or `--block-llm`) flag ensures we don't make expensive API calls, but we SHOULD have all responses in the downloaded cache.

In [ ]:
configs_to_run = [
    "config/embs_vs_llms/wild_dev_sim.yaml",
    "config/embs_vs_llms/wild_dev_sim_one_shot_t=1.yaml",
    "config/embs_vs_llms/wild_dev_sim_vec.yaml",
    "config/embs_vs_llms/wild_dev_sim_vec_vid.yaml"
]

# now relative to root
main_script = "src/main.py"

for conf in configs_to_run:
    print(f"\n{'='*50}\nRunning: {conf}\n{'='*50}")
    # Standard subprocess call, inheriting CWD (project root)
    cmd = [sys.executable, main_script, conf, "--block-llm"]
    subprocess.check_call(cmd)

## 4. Analysis
Now we analyze the results using the `ranking3` logic.
Select two methods from the dropdowns below to compare them.

In [ ]:
import ipywidgets as widgets
from analysis.llm_based import load_dfs, AnalysisArgs
from analysis import ranking3
import matplotlib.pyplot as plt
from IPython.display import display

# Load data relative to root
results_path = Path("results/upload")
df, df_z = load_dfs(results_path)
methods = sorted(df['method'].unique())

# Create widgets
method1_dropdown = widgets.Dropdown(options=methods, description='Method 1:')
method2_dropdown = widgets.Dropdown(options=methods, description='Method 2:')
metric_text = widgets.Text(value='cos_sim_mean', description='Metric:')
btn = widgets.Button(description="Run Analysis")
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()
        m1 = method1_dropdown.value
        m2 = method2_dropdown.value
        metric = metric_text.value
        
        print(f"Comparing {m1} vs {m2} on {metric}...")
        
        args = AnalysisArgs(
            method1=m1,
            method2=m2,
            metric=metric
        )
        
        ranking3.main(args)
        
        # Display plot
        result_img_path = Path("results/plots/rank_stability_Nov") / "rank_comparison_at_6_masked.png"
        if result_img_path.exists():
             img = plt.imread(str(result_img_path))
             plt.figure(figsize=(20, 10))
             plt.imshow(img)
             plt.axis('off')
             plt.show()

btn.on_click(on_click)
display(method1_dropdown, method2_dropdown, metric_text, btn, output)